# 🧩 Mini-Lab: Majority-Class Baseline

**Module: Is It Any Good? Quality, Latency, Cost** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Understand** why a dumb baseline is the first thing you need before evaluating any AI system
2. **Implement** a majority-class baseline that always predicts the most common label
3. **Compare** your AI system's accuracy against the baseline to prove it adds value

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Baseline Comparison | Judge output against a simple reference baseline (a cheap, clear alternative), not against "looks good". If you cannot beat the baseline, you have not earned complexity. |


## Why Start With a Baseline?

Imagine your AI classifier scores **82% accuracy** on a support-ticket dataset. Impressive? Not necessarily.

If **80% of the tickets are "billing"** issues, a system that *always* guesses "billing" — without reading the ticket at all — would also hit 80%. Your fancy model only beats the dumbest possible approach by 2 percentage points.

A **majority-class baseline** is the simplest baseline: always predict the most frequent label. If your AI can't beat this, it isn't adding value.

## Step 1 — Create a Small Labeled Dataset

We'll simulate a set of support tickets with human-verified labels (a "gold set") and the predictions an AI system made.

In [1]:
# Simulated gold-set: true labels and AI predictions for 20 support tickets
true_labels = [
    "billing", "billing", "billing", "billing", "billing",
    "billing", "billing", "billing", "billing", "billing",
    "billing", "billing", "billing", "billing",           # 14 billing
    "technical", "technical", "technical",                  # 3 technical
    "account", "account", "account",                       # 3 account
]

ai_predictions = [
    "billing", "billing", "billing", "billing", "billing",
    "billing", "billing", "billing", "billing", "billing",
    "billing", "billing", "technical", "account",          # 2 wrong on billing
    "technical", "technical", "billing",                   # 1 wrong on technical
    "account", "account", "billing",                      # 1 wrong on account
]

print(f"Total tickets: {len(true_labels)}")
print(f"Label distribution: billing={true_labels.count('billing')}, "
      f"technical={true_labels.count('technical')}, "
      f"account={true_labels.count('account')}")

Total tickets: 20
Label distribution: billing=14, technical=3, account=3


## Step 2 — Build the Majority-Class Baseline

Find the most common label, then create a "prediction" list that always guesses that label.

In [2]:
from collections import Counter

# Find the majority class
label_counts = Counter(true_labels)
majority_class = label_counts.most_common(1)[0][0]

print(f"Majority class: '{majority_class}' ({label_counts[majority_class]}/{len(true_labels)} tickets)")

# The baseline always predicts the majority class
baseline_predictions = [majority_class] * len(true_labels)

Majority class: 'billing' (14/20 tickets)


## Step 3 — Compare Accuracy: Baseline vs. AI

In [3]:
def accuracy(true, predicted):
    """Fraction of predictions that match the true labels."""
    correct = sum(t == p for t, p in zip(true, predicted))
    return correct / len(true)

baseline_accuracy = accuracy(true_labels, baseline_predictions) * 100
ai_accuracy       = accuracy(true_labels, ai_predictions) * 100
margin             = ai_accuracy - baseline_accuracy

print(f"Baseline accuracy (always guess '{majority_class}'): {baseline_accuracy:.1f}%")
print(f"AI system accuracy:                                  {ai_accuracy:.1f}%")
print(f"Margin over baseline:                                {margin:+.1f} percentage points")
print()

if margin > 0:
    print("✅ The AI system beats the majority-class baseline.")
else:
    print("❌ The AI system does NOT beat the baseline — it adds no value.")

Baseline accuracy (always guess 'billing'): 70.0%
AI system accuracy:                                  80.0%
Margin over baseline:                                +10.0 percentage points

✅ The AI system beats the majority-class baseline.


## Step 4 — Interpret the Result

The AI scored higher than the baseline — but look at the margin. With a skewed dataset (70% billing), the baseline is already at 70%. A small margin means the AI is barely doing better than guessing the same thing every time.

**The takeaway:** always report your AI's accuracy *relative to the baseline*, not in isolation. An 80% accuracy sounds great until you learn the baseline is 78%.

In [4]:
from IPython.display import display, Markdown
def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

report = f"""
### Baseline Evaluation Report

| Metric | Value |
|--------|-------|
| Dataset size | {len(true_labels)} tickets |
| Majority class | `{majority_class}` ({label_counts[majority_class]}/{len(true_labels)}) |
| Baseline accuracy | {baseline_accuracy:.1f}% |
| AI accuracy | {ai_accuracy:.1f}% |
| **Margin** | **{margin:+.1f} pp** |

> **Rule of thumb:** If the margin is small, invest effort in improving the
> AI system before shipping — or question whether you need AI at all for this task.
"""

md(report)


### Baseline Evaluation Report

| Metric | Value |
|--------|-------|
| Dataset size | 20 tickets |
| Majority class | `billing` (14/20) |
| Baseline accuracy | 70.0% |
| AI accuracy | 80.0% |
| **Margin** | **+10.0 pp** |

> **Rule of thumb:** If the margin is small, invest effort in improving the
> AI system before shipping — or question whether you need AI at all for this task.


## 🎯 Summary

### Key Takeaways

1. **Majority-class baseline** — the simplest possible predictor that always guesses the most frequent label; it's the floor your AI must beat
2. **Accuracy in isolation is misleading** — an 80% accuracy means nothing if the baseline is already 78%
3. **Always compare against a baseline** — report the margin (AI accuracy minus baseline accuracy) to prove your system adds real value